In [0]:
# 03_silver — Bronze → Silver 공통 모델 변환
# 계약 기준: docs/contracts/silver-schema.md

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")

CATALOG = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

def silver_id(*parts):
    """sha256(part1:part2:...) — Silver ID 생성 규칙"""
    return F.sha2(F.concat_ws(":", *parts), 256)


## 1. silver_principals
# NHI: bronze_azure_applications → principal_kind=NHI, principal_type=app_registration
# HI : bronze_identity_lifecycle_seed → principal_kind=HI, principal_type=user

In [0]:
nhi = spark.table(f"{CATALOG}.bronze.bronze_azure_applications").select(
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("app_registration")).alias("principal_id"),
    F.lit("entra_id").alias("source_system"),
    F.col("app_id").alias("source_principal_id"),
    F.lit("NHI").alias("principal_kind"),
    F.lit("app_registration").alias("principal_type"),
    F.col("display_name"),
    F.lit(None).cast("string").alias("email"),
    F.lit("active").alias("status"),
    F.lit(None).cast("date").alias("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)

status_map = F.when(F.col("employment_status") == "terminated", F.lit("terminated")) \
              .when(F.col("employment_status") == "transferred", F.lit("inactive")) \
              .otherwise(F.lit("active"))

human = spark.table(f"{CATALOG}.bronze.bronze_identity_lifecycle_seed").select(
    silver_id(F.lit("identity_seed"), F.col("user_principal_name"), F.lit("user")).alias("principal_id"),
    F.lit("identity_seed").alias("source_system"),
    F.col("user_principal_name").alias("source_principal_id"),
    F.lit("HI").alias("principal_kind"),
    F.lit("user").alias("principal_type"),
    F.coalesce(F.col("display_name"), F.col("user_principal_name")).alias("display_name"),
    F.col("user_principal_name").alias("email"),
    status_map.alias("status"),
    F.col("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_apps_raw = spark.table(f"{CATALOG}.bronze.bronze_github_apps")
github_apps = github_apps_raw.select(
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")).alias("principal_id"),
    F.lit("entra_id").alias("source_system"),
    F.col("app_id").alias("source_principal_id"),
    F.lit("NHI").alias("principal_kind"),
    F.lit("github_app").alias("principal_type"),
    F.col("display_name"),
    F.col("owner_email").alias("email"),
    F.lit("active").alias("status"),
    F.lit(None).cast("date").alias("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_owner_status = (
    F.when(F.col("owner_employment_status") == "terminated", F.lit("terminated"))
     .when(F.col("owner_employment_status").isin("inactive", "transferred", "suspended", "deactivated"), F.lit("inactive"))
     .otherwise(F.lit("active"))
)
github_owner_humans_candidate = github_apps_raw.where(F.col("owner_email").isNotNull()).select(
    silver_id(F.lit("identity_seed"), F.col("owner_email"), F.lit("user")).alias("principal_id"),
    F.lit("identity_seed").alias("source_system"),
    F.col("owner_email").alias("source_principal_id"),
    F.lit("HI").alias("principal_kind"),
    F.lit("user").alias("principal_type"),
    F.col("owner_email").alias("display_name"),
    F.col("owner_email").alias("email"),
    github_owner_status.alias("status"),
    F.col("owner_termination_date").alias("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)
github_owner_humans = github_owner_humans_candidate.join(
    human.select("principal_id").distinct(),
    on="principal_id",
    how="left_anti",
)

google_users = spark.table(f"{CATALOG}.bronze.bronze_google_workspace_users").select(
    silver_id(F.lit("identity_seed"), F.col("user_id"), F.lit("user")).alias("principal_id"),
    F.lit("identity_seed").alias("source_system"),
    F.col("user_id").alias("source_principal_id"),
    F.lit("HI").alias("principal_kind"),
    F.lit("user").alias("principal_type"),
    F.col("display_name"),
    F.col("email"),
    F.when(F.col("employment_status") == "terminated", F.lit("terminated"))
     .when(F.col("status") == "suspended", F.lit("inactive"))
     .otherwise(F.lit("active")).alias("status"),
    F.col("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)

slack_users = spark.table(f"{CATALOG}.bronze.bronze_slack_users").select(
    silver_id(F.lit("identity_seed"), F.col("user_id"), F.lit("user")).alias("principal_id"),
    F.lit("identity_seed").alias("source_system"),
    F.col("user_id").alias("source_principal_id"),
    F.lit("HI").alias("principal_kind"),
    F.lit("user").alias("principal_type"),
    F.col("display_name"),
    F.col("email"),
    F.when(F.col("employment_status") == "terminated", F.lit("terminated"))
     .when(F.col("status") == "deactivated", F.lit("inactive"))
     .otherwise(F.lit("active")).alias("status"),
    F.col("termination_date"),
    F.col("_ingestion_ts").alias("observed_at"),
)

(nhi.unionByName(human)
    .unionByName(github_apps)
    .unionByName(github_owner_humans)
    .unionByName(google_users)
    .unionByName(slack_users)
 .write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.silver.silver_principals"))
print(f"silver_principals: {spark.table(f'{CATALOG}.silver.silver_principals').count()} rows")


## 2. silver_credentials
# bound_principal_id = silver principal_id (sha256), is_active = status=active AND not expired

In [0]:
azure_creds = spark.table(f"{CATALOG}.bronze.bronze_azure_password_credentials").select(
    silver_id(
        F.lit("entra_id"),
        F.col("key_id"),
        silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("app_registration"))
    ).alias("credential_id"),
    F.lit("entra_id").alias("source_system"),
    F.col("key_id").alias("source_credential_id"),
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("app_registration")).alias("bound_principal_id"),
    F.lit("client_secret").alias("credential_type"),
    F.col("display_name"),
    F.col("status"),
    F.col("end_date_time").alias("expires_at"),
    F.when(
        (F.col("status") == "active") &
        (F.col("end_date_time").isNull() | (F.col("end_date_time") > F.col("_ingestion_ts"))),
        F.lit(True)
    ).otherwise(F.lit(False)).alias("is_active"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_creds = spark.table(f"{CATALOG}.bronze.bronze_github_credentials").select(
    silver_id(
        F.lit("entra_id"),
        F.col("credential_id"),
        silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app"))
    ).alias("credential_id"),
    F.lit("entra_id").alias("source_system"),
    F.col("credential_id").alias("source_credential_id"),
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")).alias("bound_principal_id"),
    F.when(F.col("credential_type") == "deploy_key", F.lit("deploy_key"))
     .when(F.col("credential_type") == "pat", F.lit("pat"))
     .otherwise(F.lit("unknown")).alias("credential_type"),
    F.col("display_name"),
    F.col("status"),
    F.col("expires_at"),
    F.when(
        (F.col("status") == "active") &
        (F.col("expires_at").isNull() | (F.col("expires_at") > F.col("_ingestion_ts"))),
        F.lit(True)
    ).otherwise(F.lit(False)).alias("is_active"),
    F.col("_ingestion_ts").alias("observed_at"),
)

(azure_creds.unionByName(github_creds)
 .write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.silver.silver_credentials"))
print(f"silver_credentials: {spark.table(f'{CATALOG}.silver.silver_credentials').count()} rows")


## 3. silver_assets
# asset_type=cloud_resource, asset_id=sha256

In [0]:
base_assets = spark.table(f"{CATALOG}.bronze.bronze_assets_seed").select(
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("cloud_resource")).alias("asset_id"),
    F.lit("assets_seed").alias("source_system"),
    F.col("asset_id").alias("source_asset_id"),
    F.lit("cloud_resource").alias("asset_type"),
    F.col("name").alias("display_name"),
    F.col("environment"),
    F.col("sensitivity"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_repos = spark.table(f"{CATALOG}.bronze.bronze_github_repos").select(
    silver_id(F.lit("assets_seed"), F.col("repo_id"), F.lit("repo")).alias("asset_id"),
    F.lit("assets_seed").alias("source_system"),
    F.col("repo_id").alias("source_asset_id"),
    F.lit("repo").alias("asset_type"),
    F.col("name").alias("display_name"),
    F.col("environment"),
    F.col("sensitivity"),
    F.col("_ingestion_ts").alias("observed_at"),
)

google_assets = spark.table(f"{CATALOG}.bronze.bronze_google_workspace_assets").select(
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("workspace")).alias("asset_id"),
    F.lit("assets_seed").alias("source_system"),
    F.col("asset_id").alias("source_asset_id"),
    F.lit("workspace").alias("asset_type"),
    F.col("name").alias("display_name"),
    F.col("environment"),
    F.col("sensitivity"),
    F.col("_ingestion_ts").alias("observed_at"),
)

slack_channels = spark.table(f"{CATALOG}.bronze.bronze_slack_channels").select(
    silver_id(F.lit("assets_seed"), F.col("channel_id"), F.lit("saas_app")).alias("asset_id"),
    F.lit("assets_seed").alias("source_system"),
    F.col("channel_id").alias("source_asset_id"),
    F.lit("saas_app").alias("asset_type"),
    F.col("name").alias("display_name"),
    F.col("environment"),
    F.col("sensitivity"),
    F.col("_ingestion_ts").alias("observed_at"),
)

(base_assets.unionByName(github_repos)
    .unionByName(google_assets)
    .unionByName(slack_channels)
 .write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.silver.silver_assets"))
print(f"silver_assets: {spark.table(f'{CATALOG}.silver.silver_assets').count()} rows")


## 4. silver_edges
# (a) access_edges: principal → asset, relationship=accesses_asset
# (b) owner_edges: human principal → NHI principal, relationship=owns_principal
# from_type/to_type: principal | asset (계약 기준)

In [0]:
# NHI access edge: Entra App Registration -> cloud asset
access_edges_raw = spark.table(f"{CATALOG}.bronze.bronze_access_edges_seed")
access_edges = access_edges_raw.select(
    silver_id(
        F.lit("access_seed"),
        silver_id(F.lit("entra_id"), F.col("principal_id"), F.lit("app_registration")),
        F.lit("accesses_asset"),
        silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("cloud_resource"))
    ).alias("edge_id"),
    F.lit("access_seed").alias("source_system"),
    silver_id(F.lit("entra_id"), F.col("principal_id"), F.lit("app_registration")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("cloud_resource")).alias("to_id"),
    F.lit("asset").alias("to_type"),
    F.col("relationship"),
    F.col("permission"),
    F.col("status"),
    F.col("last_used_at"),
    F.lit("bronze_access_edges_seed").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

owner_edges_raw = spark.table(f"{CATALOG}.bronze.bronze_azure_application_owners")
owner_edges = owner_edges_raw.select(
    silver_id(
        F.lit("entra_id"),
        silver_id(F.lit("identity_seed"), F.col("user_principal_name"), F.lit("user")),
        F.lit("owns_principal"),
        silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("app_registration"))
    ).alias("edge_id"),
    F.lit("entra_id").alias("source_system"),
    silver_id(F.lit("identity_seed"), F.col("user_principal_name"), F.lit("user")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("app_registration")).alias("to_id"),
    F.lit("principal").alias("to_type"),
    F.lit("owns_principal").alias("relationship"),
    F.lit(None).cast("string").alias("permission"),
    F.lit("active").alias("status"),
    F.lit(None).cast("timestamp").alias("last_used_at"),
    F.lit("bronze_azure_application_owners").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_owner_edges = spark.table(f"{CATALOG}.bronze.bronze_github_apps").where(F.col("owner_email").isNotNull()).select(
    silver_id(
        F.lit("entra_id"),
        silver_id(F.lit("identity_seed"), F.col("owner_email"), F.lit("user")),
        F.lit("owns_principal"),
        silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app"))
    ).alias("edge_id"),
    F.lit("entra_id").alias("source_system"),
    silver_id(F.lit("identity_seed"), F.col("owner_email"), F.lit("user")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")).alias("to_id"),
    F.lit("principal").alias("to_type"),
    F.lit("owns_principal").alias("relationship"),
    F.lit(None).cast("string").alias("permission"),
    F.lit("active").alias("status"),
    F.lit(None).cast("timestamp").alias("last_used_at"),
    F.lit("bronze_github_apps").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_credential_edges = spark.table(f"{CATALOG}.bronze.bronze_github_credentials").select(
    silver_id(
        F.lit("access_seed"),
        silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")),
        F.lit("owns_credential"),
        silver_id(F.lit("entra_id"), F.col("credential_id"), silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")))
    ).alias("edge_id"),
    F.lit("access_seed").alias("source_system"),
    silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("entra_id"), F.col("credential_id"), silver_id(F.lit("entra_id"), F.col("app_id"), F.lit("github_app"))).alias("to_id"),
    F.lit("credential").alias("to_type"),
    F.lit("owns_credential").alias("relationship"),
    F.lit(None).cast("string").alias("permission"),
    F.col("status"),
    F.lit(None).cast("timestamp").alias("last_used_at"),
    F.lit("bronze_github_credentials").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

github_access_edges = spark.table(f"{CATALOG}.bronze.bronze_github_access_edges").select(
    silver_id(
        F.lit("access_seed"),
        silver_id(F.lit("entra_id"), F.col("principal_id"), F.lit("github_app")),
        F.lit("accesses_asset"),
        silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("repo"))
    ).alias("edge_id"),
    F.lit("access_seed").alias("source_system"),
    silver_id(F.lit("entra_id"), F.col("principal_id"), F.lit("github_app")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("repo")).alias("to_id"),
    F.lit("asset").alias("to_type"),
    F.col("relationship"),
    F.col("permission"),
    F.col("status"),
    F.col("last_used_at"),
    F.lit("bronze_github_access_edges").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

google_access_edges = spark.table(f"{CATALOG}.bronze.bronze_google_workspace_access_edges").select(
    silver_id(
        F.lit("access_seed"),
        silver_id(F.lit("identity_seed"), F.col("principal_id"), F.lit("user")),
        F.lit("accesses_asset"),
        silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("workspace"))
    ).alias("edge_id"),
    F.lit("access_seed").alias("source_system"),
    silver_id(F.lit("identity_seed"), F.col("principal_id"), F.lit("user")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("workspace")).alias("to_id"),
    F.lit("asset").alias("to_type"),
    F.col("relationship"),
    F.col("permission"),
    F.col("status"),
    F.col("last_used_at"),
    F.lit("bronze_google_workspace_access_edges").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

slack_access_edges = spark.table(f"{CATALOG}.bronze.bronze_slack_access_edges").select(
    silver_id(
        F.lit("access_seed"),
        silver_id(F.lit("identity_seed"), F.col("principal_id"), F.lit("user")),
        F.lit("accesses_asset"),
        silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("saas_app"))
    ).alias("edge_id"),
    F.lit("access_seed").alias("source_system"),
    silver_id(F.lit("identity_seed"), F.col("principal_id"), F.lit("user")).alias("from_id"),
    F.lit("principal").alias("from_type"),
    silver_id(F.lit("assets_seed"), F.col("asset_id"), F.lit("saas_app")).alias("to_id"),
    F.lit("asset").alias("to_type"),
    F.col("relationship"),
    F.col("permission"),
    F.col("status"),
    F.col("last_used_at"),
    F.lit("bronze_slack_access_edges").alias("evidence_source"),
    F.col("_ingestion_ts").alias("observed_at"),
)

(access_edges.unionByName(owner_edges)
    .unionByName(github_owner_edges)
    .unionByName(github_credential_edges)
    .unionByName(github_access_edges)
    .unionByName(google_access_edges)
    .unionByName(slack_access_edges)
 .write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.silver.silver_edges"))
print(f"silver_edges: {spark.table(f'{CATALOG}.silver.silver_edges').count()} rows")


## 검증

In [0]:
silver_tables = [
    "silver_principals",
    "silver_credentials",
    "silver_assets",
    "silver_edges",
]

summary = ["silver smoke test passed"]
for table in silver_tables:
    actual = spark.table(f"{CATALOG}.silver.{table}").count()
    assert actual > 0, f"{table} has no rows"
    line = f"{table}: {actual} rows"
    print(line)
    summary.append(line)

required_columns = {
    "silver_principals": ["principal_id", "source_system", "source_principal_id", "principal_kind", "principal_type", "status", "observed_at"],
    "silver_credentials": ["credential_id", "source_system", "source_credential_id", "bound_principal_id", "credential_type", "status", "is_active", "observed_at"],
    "silver_assets": ["asset_id", "source_system", "source_asset_id", "asset_type", "environment", "sensitivity", "observed_at"],
    "silver_edges": ["edge_id", "source_system", "from_id", "from_type", "to_id", "to_type", "relationship", "status", "observed_at"],
}

for table, columns in required_columns.items():
    condition = " OR ".join([f"{col} IS NULL" for col in columns])
    missing = spark.table(f"{CATALOG}.silver.{table}").where(condition).count()
    assert missing == 0, f"{table} has null required key columns: {missing} rows"
    summary.append(f"{table}.required_nulls: {missing}")

principals = spark.table(f"{CATALOG}.silver.silver_principals")
credentials = spark.table(f"{CATALOG}.silver.silver_credentials")
assets = spark.table(f"{CATALOG}.silver.silver_assets")
edges = spark.table(f"{CATALOG}.silver.silver_edges")

active_credentials = credentials.where("is_active = true").count()
assert active_credentials > 0, "expected at least one active credential"
summary.append(f"active_credentials: {active_credentials}")

missing_credential_principals = credentials.join(
    principals,
    credentials.bound_principal_id == principals.principal_id,
    "left_anti",
).count()
assert missing_credential_principals == 0, f"credentials without principal: {missing_credential_principals}"
summary.append(f"credentials_without_principal: {missing_credential_principals}")

principal_from_edges = edges.where("from_type = 'principal'")
missing_from_principals = principal_from_edges.join(
    principals,
    principal_from_edges.from_id == principals.principal_id,
    "left_anti",
).count()
assert missing_from_principals == 0, f"principal edges without from principal: {missing_from_principals}"
summary.append(f"principal_edges_without_from_principal: {missing_from_principals}")

principal_to_edges = edges.where("to_type = 'principal'")
missing_to_principals = principal_to_edges.join(
    principals,
    principal_to_edges.to_id == principals.principal_id,
    "left_anti",
).count()
assert missing_to_principals == 0, f"principal edges without to principal: {missing_to_principals}"
summary.append(f"principal_edges_without_to_principal: {missing_to_principals}")

asset_to_edges = edges.where("to_type = 'asset'")
missing_to_assets = asset_to_edges.join(
    assets,
    asset_to_edges.to_id == assets.asset_id,
    "left_anti",
).count()
assert missing_to_assets == 0, f"asset edges without to asset: {missing_to_assets}"
summary.append(f"asset_edges_without_to_asset: {missing_to_assets}")

credential_to_edges = edges.where("to_type = 'credential'")
missing_to_credentials = credential_to_edges.join(
    credentials,
    credential_to_edges.to_id == credentials.credential_id,
    "left_anti",
).count()
assert missing_to_credentials == 0, f"credential edges without to credential: {missing_to_credentials}"
summary.append(f"credential_edges_without_to_credential: {missing_to_credentials}")

github_owner_expected = spark.table(f"{CATALOG}.bronze.bronze_github_apps").where("owner_email IS NOT NULL").count()
github_owner_edge_rows = (
    edges.alias("e")
    .join(principals.alias("owner"), F.col("e.from_id") == F.col("owner.principal_id"), "inner")
    .join(principals.alias("app"), F.col("e.to_id") == F.col("app.principal_id"), "inner")
    .where(
        (F.col("e.relationship") == F.lit("owns_principal"))
        & (F.col("e.evidence_source") == F.lit("bronze_github_apps"))
        & (F.col("owner.principal_kind") == F.lit("HI"))
        & (F.col("owner.status").isin("terminated", "inactive", "active"))
        & (F.col("app.principal_type") == F.lit("github_app"))
    )
    .count()
)
assert github_owner_edge_rows == github_owner_expected, (
    f"github owner edges mismatch: expected {github_owner_expected}, actual {github_owner_edge_rows}"
)
summary.append(f"github_owner_edges: {github_owner_edge_rows}")

spark.table(f"{CATALOG}.silver.silver_principals").select("principal_id","principal_kind","principal_type","status").show(truncate=False)
print("silver transform complete")
dbutils.notebook.exit("\n".join(summary))
